# Création d'une CLI de stéganographie

La [stéganographie](https://fr.wikipedia.org/wiki/St%C3%A9ganographie) regroupe les techniques permettant de dissimuler un message. Elle se distingue de la cryptographie, qui ne cherche pas à cacher un message mais à le rendre incompréhensible pour qui ne possède pas la clé nécessaire.

La méthode que nous allons implémenter dans ces travaux pratiques est bien connue : elle consiste à [modifier le bit de poids faible de chaque pixel d'une image](https://fr.wikipedia.org/wiki/St%C3%A9ganographie#Modification_des_bits_de_poids_faible_d'une_image). Cette modification n'est pas visible à l'œil nu (même si elle peut être détectée par une analyse statistique).

## Environnement de travail

Pour les TPs, nous utiliserons GitHub Codespaces, qui fournit des instances de VSCode dans des conteneurs. La version gratuite permet 30 heures d'utilisation par mois avec 15 Go de stockage. C'est largement suffisant pour ces travaux pratiques.

Si vous n'avez pas de compte GitHub, commencez par en créer un [ici](https://github.com/signup).

[![Accéder à l'environnement de travail](https://github.com/codespaces/badge.svg)](https://github.com/codespaces/new?hide_repo_select=true&ref=main&repo=572729501&machine=basicLinux32gb&devcontainer_path=.devcontainer%2Fdevcontainer.json&location=WestEurope)

## Étape 1 ⋅ Fonctions de conversion de chaînes de caractères en bits et inversement

Dans un premier temps, nous allons nous concentrer sur la conversion de chaînes de caractères en bits (et inversement).

Une première fonction, `encode_string`, prendra en argument une chaîne de caractères et renverra un tableau numpy des bits correspondants (dans l'encodage utf8).

Un tableau numpy est une structure de données homogène (éléments de même type), que nous utiliserons dans ces travaux pratiques comme une liste très améliorée.

Par exemple, pour la chaîne `"ABC"`, on attend le tableau de sortie suivant :

```python
numpy.array([0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1])
```

En effet, `A` s'encode `01000001` en utf8, `B` `01000010` et `C` `01000011`. Le tableau attendu est simplement la concaténation de ces bits.

Pour coder cette fonction, voici quelques fonctions utiles :
- [`str.encode`](https://docs.python.org/fr/3/library/stdtypes.html#str.encode), qui renvoie un objet `bytes` (lui-même un itérable d'octets représentés par des entiers)
- [`numpy.unpackbits`](https://numpy.org/doc/stable/reference/generated/numpy.unpackbits.html), que vous pouvez appliquer à un tableau numpy créé au préalable à partir des octets représentés par des entiers obtenus depuis l'objet `bytes`. Pour créer le tableau numpy, vous pouvez utiliser [`numpy.array`](https://numpy.org/doc/stable/reference/generated/numpy.array.html) ou [`numpy.fromiter`](https://numpy.org/doc/stable/reference/generated/numpy.fromiter.html) en précisant `dtype="uint8"` lors de l'appel. Cet argument vous permettra de représenter les valeurs des octets avec le type le plus adapté.

Voici la signature attendue de la fonction :

```python
def encode_string(string: str) -> numpy.ndarray:
```

La fonction `decode_bits` se chargera de la transformation inverse. Les fonctions à utiliser sont les inverses de celles nécessaires pour `encode_string` :
- [`bytes.decode`](https://docs.python.org/fr/3/library/stdtypes.html#bytes.decode), qui renvoie une chaîne de caractères
- [`numpy.packbits`](https://numpy.org/doc/stable/reference/generated/numpy.packbits.html)

Voici la signature attendue de la fonction :

```python
def decode_bits(bits: numpy.ndarray) -> str:
```

Consignes :

- Créez le fichier `tests/test_string_conversion.py` et écrivez au moins un cas de test par fonction, en utilisant l'exemple donné si besoin. Pour comparer deux tableaux numpy, vous pouvez utiliser la fonction dédiée [`numpy.testing.assert_array_equal`](https://numpy.org/doc/stable/reference/generated/numpy.testing.assert_array_equal.html).
- Créez le fichier `steganosaure/string_conversion.py` et implémentez-y ces deux fonctions pour que les tests passent.
- Pouvez-vous concevoir un test qui s'assure que les deux fonctions sont bien inverses l'une de l'autre ?

## Étape 2 ⋅ Fonctions de chargement et de sauvegarde d'images

Nous allons maintenant coder deux fonctions importantes : l'une chargera une image, l'autre la sauvegardera.

**Attention, pour que notre algorithme fonctionne, une image sauvegardée puis rechargée doit être identique à l'image d'origine.**

La fonction de chargement devra charger une image dans un tableau numpy dont les éléments sont des entiers 8 bits (pour cela, vous pouvez utiliser le paquet [`scikit-image`](https://scikit-image.org/), et en particulier les fonctions [`skimage.io.imread`](https://scikit-image.org/docs/stable/api/skimage.io.html#skimage.io.imread) et [`skimage.util.img_as_ubyte`](https://scikit-image.org/docs/stable/api/skimage.util.html#skimage.util.img_as_ubyte)). Voici la signature attendue :

```python
def load_image(image_path: pathlib.Path) -> numpy.ndarray:
```

La fonction de sauvegarde devra sauvegarder un tableau numpy au chemin indiqué. Notez que pour que l'image rechargée soit équivalente à l'image sauvegardée tout en gardant une bonne qualité, il est préférable de sauvegarder les images au format `.png`. Vous pouvez utiliser la fonction [`skimage.io.imsave`](https://scikit-image.org/docs/stable/api/skimage.io.html#skimage.io.imsave). Il sera utile de lui passer les arguments `plugin="pil"` et `check_contrast=False`. Voici la signature attendue :

```python
def save_image(image: numpy.ndarray, image_path: pathlib.Path) -> None:
```

Consignes :

- Créez le fichier `tests/test_image_io.py` et écrivez au moins un cas de test qui s'assure que le point crucial est respecté (une image sauvegardée puis chargée est identique à l'image d'origine).
- Créez le fichier `steganosaure/image_io.py` et implémentez-y ces deux fonctions pour que le ou les tests passent.

## Étape 3 ⋅ Fonctions de manipulation de bits

Nous allons maintenant implémenter le cœur de l'algorithme : modifier le bit de poids faible de chaque pixel pour y stocker un bit d'information de la chaîne à cacher.

Petit rappel sur la manipulation de bits :

```python
0b01 & 0b11 == 0b01  # ET bit à bit : le bit résultat vaut 1 si les deux bits valent 1
0b01 | 0b11 == 0b11  # OU bit à bit : le bit résultat vaut 1 si au moins un bit vaut 1
0b11 >> 1 == 0b01    # décalage à droite, complète avec des 0
0b11 << 1 == 0b10    # décalage à gauche, complète avec des 0
```

Ces représentations sont équivalentes à leur notation décimale :

```python
1 & 3 == 1
1 | 3 == 3
3 >> 1 == 1
3 << 1 == 2
```

Pour cela, nous allons implémenter deux fonctions. La première, `retrieve_least_significant_bits`, renverra un tableau numpy de même taille que son argument mais ne contenant que des 0 ou des 1 : les bits de poids faible des nombres contenus dans le tableau en argument.

Pour écrire cette fonction, n'hésitez pas à tirer parti du fait qu'on peut appliquer directement une opération à un tableau numpy (y compris les opérations bit à bit) :

```python
>>> 3 & 1
1
>>> 2 & 1
0
>>> numpy.array([3, 2]) & 1
array([1, 0])
```

Voici sa signature attendue :

```python
def retrieve_least_significant_bits(array: numpy.ndarray) -> numpy.ndarray:
```

La seconde fonction à implémenter, `modify_least_significant_bits`, modifiera les bits de poids faible de `image` pour qu'ils correspondent au tableau `message`. Voici sa signature attendue :

```python
def modify_least_significant_bits(message: numpy.ndarray, image: numpy.ndarray) -> numpy.ndarray:
```

Consignes :

- Créez le fichier `tests/test_bits_manipulation.py` et écrivez au moins un cas de test par fonction.
- Créez le fichier `steganosaure/bits_manipulation.py` et implémentez-y ces deux fonctions pour que les tests passent.

## Étape 4 ⋅ Fonctions de manipulation de tableaux numpy

La troisième fonction à implémenter est `pad_array`. Son but est de prétraiter les arguments de `modify_least_significant_bits` pour qu'ils aient la même taille. Cette fonction doit renvoyer un `small_array` (qui sera notre message à cacher) de même taille que le `big_array` (qui sera l'image dans laquelle le message est dissimulé). Pour cela, `small_array` sera complété avec des zéros. Vous pouvez utiliser [`numpy.pad`](https://numpy.org/doc/stable/reference/generated/numpy.pad.html) pour l'implémentation.

Voici sa signature attendue :

```python
def pad_array(small_array: numpy.ndarray, big_array: numpy.ndarray) -> numpy.ndarray:
```

## Étape 5 ⋅ Implémentation de la stéganographie

Nous avons maintenant terminé la phase préparatoire et disposons de toutes les fonctions nécessaires pour implémenter la stéganographie.

La première fonction de notre outil, `encrypt`, permettra à l'utilisateur de cacher un message dans une image. Avant d'appliquer les méthodes déjà développées, il faudra utiliser [`numpy.reshape`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) pour obtenir toutes les valeurs de pixels dans un tableau à une dimension.

Voici sa signature attendue :

```python
def encrypt(message: str, input_image_path: pathlib.Path, output_image_path: pathlib.Path) -> None:
```

La seconde fonction de notre outil, `decrypt`, permettra à l'utilisateur de retrouver le message précédemment caché dans une image. Voici sa signature attendue :

```python
def decrypt(image_path: pathlib.Path) -> str:
```

Consignes :

- Créez le fichier `tests/test_steganography.py` et écrivez au moins un cas de test qui vérifie le bon fonctionnement des deux fonctions ensemble.
- Créez le fichier `steganosaure/steganography.py` et implémentez-y ces deux fonctions pour que les tests passent.

### Étape 6 ⋅ Création d'un point d'entrée

Pour cette étape, nous allons créer un point d'entrée en ligne de commande pour notre projet `steganosaure`.

1. **Création du fichier `cli.py`** : créez ce fichier dans votre dossier source (`steganosaure`).

2. **Implémentation de la fonction `main`** : dans ce fichier, écrivez une fonction `main` qui ne prend pas d'argument et affiche simplement `Hello World`.

3. **Modification du fichier `pyproject.toml`** : éditez ce fichier pour configurer la commande `steganosaure` afin qu'elle exécute la fonction `main` de `cli.py`. Ajoutez la ligne suivante sous `[tool.poetry.scripts]` :

   ```ini
   steganosaure = 'steganosaure.cli:main'
   ```

   Pensez à remplacer `steganosaure` par le nom de votre projet et `cli` par le nom du fichier qui contient la fonction `main`.

4. **Installation des dépendances** : lancez `poetry install` pour appliquer les modifications du point d'entrée et préparer le projet à l'utilisation.

5. **Test du point d'entrée** : testez le point d'entrée en exécutant la commande `steganosaure`. Elle doit afficher `Hello World` si tout est correctement configuré.

Consignes :

- Suivez bien toutes les étapes dans l'ordre pour garantir le bon fonctionnement du point d'entrée.
- Une fois le point d'entrée configuré, vous pouvez personnaliser la fonction `main` pour qu'elle effectue des actions propres à votre projet.

In [ ]:
!poetry install

In [ ]:
!steganosaure

### Étape 7 ⋅ Ajout d'un parseur

Dans cette étape, nous allons modifier la fonction `main` du fichier `cli.py` pour qu'elle gère une commande (`encrypt` ou `decrypt`) et les arguments nécessaires :

- Pour `encrypt` : `message`, `input_image_path` et `output_image_path`
- Pour `decrypt` : `image_path`

Nous nous inspirerons pour cette étape de l'exemple des slides, très proche de ce que nous cherchons à faire ici.

Une fois ces arguments parsés, nous les relierons aux fonctions correspondantes.

Consignes :

- Modifiez la fonction `main` du fichier `cli.py` pour y ajouter la gestion et le parsing des arguments de la ligne de commande.
- Utilisez une librairie de parsing telle que `argparse` ou `click` pour faciliter la gestion des commandes et des arguments.
- Reliez les arguments parsés aux fonctions `encrypt` et `decrypt` déjà implémentées dans votre projet.

## Solution

[Dépôt du TP](https://github.com/mlambda/tp-steganosaure/tree/solution).